In [30]:
import re
import time
import json
import html
import os
import requests
import pandas as pd

# ช่วงเวลาข่าวตามเวลาไทย
START_DATE = "2026-10-13"
END_DATE   = "2026-10-6"

# API และข้อจำกัดการค้น
BASE_URL = "https://api.thaipbs.net"
SITE = "news"
MAX_PAGES = 300
STOP_AFTER_OLD_PAGES = 2
MAX_DETAIL = 500

# ลดความถี่ในการเรียก API
REQUEST_DELAY = 0.4

# Cache ป้องกันการดึงบทความเดิมซ้ำ
CACHE_FILE = "thaipbs_cache.json"

TZ = "Asia/Bangkok"

START = pd.Timestamp(START_DATE, tz=TZ) if START_DATE else None
END_EXCLUSIVE = (
    pd.Timestamp(END_DATE, tz=TZ) + pd.Timedelta(days=1)
    if END_DATE else pd.Timestamp.now(tz=TZ) + pd.Timedelta(days=1)
)

session = requests.Session()
session.headers.update({
    "User-Agent": "gold-news-research/1.0",
    "Accept": "application/json",
})

print(f"ช่วงข่าว: {START_DATE or 'ไม่จำกัด'} ถึง {END_DATE or 'วันนี้'}")

ช่วงข่าว: 2026-10-13 ถึง 2026-10-6


In [31]:
def api_get(path, params=None, retries=3):
    """เรียก Thai PBS API และคืน JSON"""
    url = path if path.startswith("http") else f"{BASE_URL}{path}"

    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=25)

            # เคารพ rate limit หาก API แจ้ง HTTP 429
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", 5))
                time.sleep(min(wait, 60))
                continue

            response.raise_for_status()
            data = response.json()
            time.sleep(REQUEST_DELAY)
            return data

        except (requests.RequestException, ValueError) as exc:
            if attempt == retries - 1:
                print(f"API error: {path} | {exc}")
                return None

            time.sleep(attempt + 1)

    return None


# โหลด Cache เก่าถ้ามี
try:
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        DETAIL_CACHE = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    DETAIL_CACHE = {}


def save_cache():
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(DETAIL_CACHE, f, ensure_ascii=False)


def get_detail(post):
    """ดึงรายละเอียดข่าวด้วย restUrl หรือ ID"""
    path = post.get("restUrl") or f"/v1/{SITE}/posts/{post.get('id')}"

    if path in DETAIL_CACHE:
        return DETAIL_CACHE[path]

    data = api_get(path)

    if not isinstance(data, dict):
        return None

    if isinstance(data.get("data"), dict) and "title" not in data:
        data = data["data"]

    DETAIL_CACHE[path] = data
    return data

In [32]:
def to_bkk(value):
    if value is None or value == "":
        return pd.NaT

    try:
        ts = pd.to_datetime(value, errors="coerce")

        if pd.isna(ts):
            return pd.NaT

        if ts.tzinfo is None:
            return ts.tz_localize(TZ)

        return ts.tz_convert(TZ)

    except (TypeError, ValueError):
        return pd.NaT


def post_date(obj):
    if not isinstance(obj, dict):
        return pd.NaT

    for key in (
        "publishTime", "publishedAt", "published_at",
        "createTime", "date"
    ):
        ts = to_bkk(obj.get(key))
        if pd.notna(ts):
            return ts

    return pd.NaT


def clean_article(raw):
    """HTML -> ข้อความที่อ่านและนำไปวิเคราะห์ต่อได้"""
    if raw is None:
        return ""

    if not isinstance(raw, str):
        raw = json.dumps(raw, ensure_ascii=False)

    text = re.sub(
        r"(?is)<(script|style)\b[^>]*>.*?</\1>",
        " ",
        raw
    )

    text = re.sub(
        r"(?i)<br\s*/?>|</(p|div|li|h[1-6]|tr|section|article)>",
        "\n",
        text
    )

    text = re.sub(r"(?s)<[^>]+>", " ", text)
    text = html.unescape(text).replace("\xa0", " ")

    lines = [
        re.sub(r"[ \t]+", " ", line).strip()
        for line in text.splitlines()
    ]

    return "\n".join(line for line in lines if line)


GOLD_DIRECT_RE = re.compile(
    r"ราคาทอง|ทองคำ|ทองแท่ง|ทองรูปพรรณ|ทองสปอต|gold\s*spot|xauusd",
    re.I
)

GOLD_CONTEXT = (
    "ราคา", "พุ่ง", "ร่วง", "ดิ่ง", "บวก", "ลบ",
    "ขึ้น", "ลง", "เฟด", "ตลาด", "ซื้อ", "ขาย",
    "ดอลลาร์", "spot"
)


def is_gold_title(title):
    """คัดจากชื่อข่าว ไม่ใช้การค้น substring ของคำว่าทองอย่างเดียว"""
    title = str(title or "").lower()

    if GOLD_DIRECT_RE.search(title):
        return True

    return (
        "ทอง" in title
        and any(word in title for word in GOLD_CONTEXT)
    )

In [33]:
def scan_gold_posts():
    candidates = {}
    older_streak = 0
    pages_scanned = 0

    for page in range(1, MAX_PAGES + 1):
        payload = api_get(
            f"/v1/{SITE}/posts",
            params={"page": page, "limit": 50}
        )

        if not isinstance(payload, dict):
            break

        posts = payload.get("data") or []

        if not posts:
            print(f"หยุดที่หน้า {page}: ไม่มีรายการข่าว")
            break

        pages_scanned += 1
        page_dates = []

        for post in posts:
            ts = post_date(post)

            if pd.notna(ts):
                page_dates.append(ts)

                if START is not None and ts < START:
                    continue

                if ts >= END_EXCLUSIVE:
                    continue

            elif not KEEP_UNDATED:
                # ถ้าไม่มีวันที่ ให้ข้ามเพื่อไม่ให้ปะปนใน Dataset
                continue

            title = post.get("title", "")

            if not is_gold_title(title):
                continue

            key = str(post.get("id") or post.get("canonical") or "")
            if key:
                candidates.setdefault(key, post)

        # หยุดเมื่อหน้าเก่ากว่าวันเริ่มต้นติดต่อกันครบตามกำหนด
        if (
            START is not None
            and len(page_dates) == len(posts)
            and max(page_dates) < START
        ):
            older_streak += 1
            if older_streak >= STOP_AFTER_OLD_PAGES:
                break
        else:
            older_streak = 0

        if page % 10 == 0:
            print(
                f"หน้า {page}: อ่าน {pages_scanned} หน้า "
                f"| พบผู้สมัคร {len(candidates)} ข่าว"
            )

    result = list(candidates.values())

    print(f"\nอ่านทั้งหมด {pages_scanned} หน้า")
    print(f"พบข่าวทองคำเบื้องต้น {len(result)} ข่าว")

    return result

In [34]:
REPORT_RE = re.compile(
    r"สรุปราคา|เช้านี้|ปิดตลาด|เปิดตลาด|ราคาทอง.*วันนี้"
)

ANALYSIS_RE = re.compile(
    r"จับตา|ปัจจัย|วิเคราะห์|แนวโน้ม|คาด|ลุ้น|กด|หนุน|"
    r"เฟด|ดอกเบี้ย|ธนาคารกลาง|ดอลลาร์|บอนด์"
)

UP_WORDS = ("พุ่ง", "ขึ้น", "บวก", "ทะลุ", "นิวไฮ", "ฟื้น")
DOWN_WORDS = ("ดิ่ง", "ร่วง", "ลบ", "ลง", "หลุด", "อ่อน")


def analyze_title(title):
    title = str(title or "")
    direction = None
    change = None

    m = re.search(r"([+\-–−＋])\s*(\d[\d,]*)\s*บาท", title)

    if m:
        amount = int(m.group(2).replace(",", ""))
        direction = "up" if m.group(1) in "+＋" else "down"
        change = amount if direction == "up" else -amount

    else:
        m = re.search(r"(บวก|ลบ)\s*(\d[\d,]*)\s*บาท", title)

        if m:
            amount = int(m.group(2).replace(",", ""))
            change = amount if m.group(1) == "บวก" else -amount
            direction = "up" if change > 0 else "down"

    if "ไม่เปลี่ยนแปลง" in title:
        direction, change = "flat", 0

    elif change is None:
        has_up = any(w in title for w in UP_WORDS)
        has_down = any(w in title for w in DOWN_WORDS)

        if has_up != has_down:
            direction = "up" if has_up else "down"

    if REPORT_RE.search(title):
        news_type = "price_report"
    elif ANALYSIS_RE.search(title):
        news_type = "market_analysis"
    else:
        news_type = "other_gold"

    report_type = None

    if news_type == "price_report":
        if re.search(r"สรุปราคา|ปิดตลาด", title):
            report_type = "close"
        elif re.search(r"เช้านี้|เปิดตลาด", title):
            report_type = "morning"
        else:
            report_type = "intraday"

    return {
        "news_type": news_type,
        "report_type": report_type,
        "direction": direction,
        "change_baht": change,
    }


def tag_names(values):
    result = []

    for item in values or []:
        if isinstance(item, dict):
            name = item.get("title") or item.get("name") or ""
        else:
            name = str(item)

        if name:
            result.append(name)

    return ", ".join(result)


def build_dataframe(posts):
    rows = []

    if len(posts) > MAX_DETAIL:
        print(f"จำกัดการดึงบทความเต็มที่ {MAX_DETAIL} ข่าว")
        posts = posts[:MAX_DETAIL]

    for i, post in enumerate(posts, start=1):
        detail = get_detail(post) or {}

        title = post.get("title") or detail.get("title") or ""

        raw = (
            detail.get("content")
            or detail.get("abstract")
            or post.get("abstract")
            or ""
        )

        article = clean_article(raw)
        published = post_date(post)

        if pd.isna(published):
            published = post_date(detail)

        # ตรวจช่วงวันที่อีกครั้งหลังได้รายละเอียด
        if pd.notna(published):
            if START is not None and published < START:
                continue
            if published >= END_EXCLUSIVE:
                continue
        elif not KEEP_UNDATED:
            continue

        features = analyze_title(title)

        rows.append({
            "published": published,
            "site": SITE,
            "id": post.get("id") or detail.get("id"),
            "title": title,
            **features,
            "tags": tag_names(post.get("tags")),
            "categories": tag_names(post.get("categories")),
            "canonical": (
                post.get("canonical")
                or detail.get("canonical")
            ),
            "body_source": (
                "content" if detail.get("content") else "abstract"
            ),
            "article_chars": len(article),
            "article": article,
        })

        # บันทึก Cache เป็นระยะ เผื่อ Notebook หยุดกลางทาง
        if i % 20 == 0:
            save_cache()
            print(f"ดึงรายละเอียดแล้ว {i}/{len(posts)} ข่าว")

    save_cache()

    df = pd.DataFrame(rows)

    if not df.empty:
        df = (
            df.sort_values("published", ascending=False, na_position="last")
              .drop_duplicates(subset=["id"])
              .reset_index(drop=True)
        )

    return df

In [35]:
posts = scan_gold_posts()
df = build_dataframe(posts)

print(f"\nข่าวที่ได้หลังตรวจข้อมูล: {len(df)} ข่าว")

if df.empty:
    print("ไม่พบข่าวในช่วงวันที่ที่กำหนด")
else:
    display_cols = [
        "published",
        "news_type",
        "report_type",
        "title",
        "direction",
        "change_baht",
        "body_source",
        "article_chars",
        "canonical",
    ]

    display(df[display_cols])

    # Export ทั้ง metadata และบทความเต็ม
    df_export = df.copy()

    df_export["published"] = df_export["published"].map(
        lambda x: (
            x.strftime("%Y-%m-%d %H:%M:%S%z")
            if pd.notna(x) else ""
        )
    )

    output_file = "thaipbs_gold_news.csv"

    df_export.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nบันทึกแล้ว: {output_file}")

หน้า 10: อ่าน 10 หน้า | พบผู้สมัคร 0 ข่าว
หน้า 20: อ่าน 20 หน้า | พบผู้สมัคร 0 ข่าว
หน้า 30: อ่าน 30 หน้า | พบผู้สมัคร 0 ข่าว
หน้า 40: อ่าน 40 หน้า | พบผู้สมัคร 0 ข่าว

อ่านทั้งหมด 45 หน้า
พบข่าวทองคำเบื้องต้น 0 ข่าว

ข่าวที่ได้หลังตรวจข้อมูล: 0 ข่าว
ไม่พบข่าวในช่วงวันที่ที่กำหนด
